# KFP チャレンジラボ 2

このチャレンジラボでは、チャレンジラボ 1 で開発したパイプラインを拡張し、バッチ予測を追加します。

**学習目標:**
1. `google-cloud-pipeline-components` の取り扱い方法を学ぶ
1. KFP パイプラインから Agent Platform のバッチ予測を使用する方法を学ぶ

## セットアップ

In [ ]:
from datetime import datetime

from google.cloud import aiplatform

%load_ext google.cloud.bigquery

In [ ]:
REGION = "us-central1"
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]

### トレーナー イメージのビルド

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "trainer_image_covertype"
IMAGE_TAG = "latest"
TRAINING_CONTAINER_IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"
TRAINING_CONTAINER_IMAGE_URI

このノートブックでは、トレーニング コンテナが `us-docker.pkg.dev/{PROJECT_ID}/asl-artifact-repo/trainer_image_covertype` にすでに作成されていることを前提としています。以下のコマンドで存在するかどうかを確認できます。


In [ ]:
!gcloud artifacts docker images list \
--include-tags us-docker.pkg.dev/$PROJECT_ID/$ARTIFACT_REGISTRY_DIR/$IMAGE_NAME \
--filter TAGS:$IMAGE_TAG

イメージが存在しない場合は、下のコメントアウトを解除してビルドしてください。

In [ ]:
# !gcloud builds submit --timeout 15m --tag $TRAINING_CONTAINER_IMAGE_URI trainer_image

モデルのサービング時に、トレーニング時に使用した ML フレームワークのバージョンと一致させるために、以下のサービング コンテナをパイプラインに提供する必要があります。

In [ ]:
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
)

**注:** トレーニング ML フレームワークのバージョンを変更する場合は、バージョンが一致するサービング コンテナを指定する必要があります（[予測用の構築済みコンテナ](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers) を参照）。

## チャレンジラボ 2

このラボでは、[challenge_lab_1.ipynb](./challenge_lab_1.ipynb) で開発したパイプラインを拡張し、バッチ予測用のコンポーネントを追加します。<br>

この機能を追加するために、[`pipeline.py`](./pipeline/pipeline.py) を以下のように変更しましょう。

[pipeline/pipeline.py](./pipeline/pipeline.py) を開き、次の要素を更新します。
- **必要なオブジェクトのインポート**: 必要なモジュールのインポート文を追加します。
- **バッチ予測コンポーネントの追加**: パイプライン定義にバッチ予測コンポーネントを追加します。
  - `instances_format` と `predictions_format` の両方に BigQuery を使用します。
  - `bigquery_source_input_uri` 引数に `f"bq://{PROJECT_ID}.covertype_dataset.validation"` を使用します。
  - `bigquery_destination_output_uri` 引数に `f"bq://{PROJECT_ID}.covertype_dataset.batch_prediction_result"` を使用します。
  - ラベルを除外するために、`excluded_fields` 引数に `"Cover_Type"` を指定します。
  - [API ドキュメント](https://google-cloud-pipeline-components.readthedocs.io/en/google-cloud-pipeline-components-2.8.0/api/v1/batch_predict_job.html?h=modelbatchpredictop#) に従ってその他の必要な引数を指定します。

**ヒント: `TODO 2` で検索して、更新が必要なセクションを特定してください。**

### 参考資料:
- 事前構築済みバッチ予測コンポーネント: https://google-cloud-pipeline-components.readthedocs.io/en/google-cloud-pipeline-components-2.8.0/api/v1/batch_predict_job.html?h=modelbatchpredictop#
- Gemini Enterprise Agent Platform バッチ予測: https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions/get-batch-predictions#all-other-containers_4

### 期待される結果
KFP パイプライン DAG（ハイライトされたセクションを拡張）:

<img src="https://github.com/GoogleCloudPlatform/asl-ml-immersion/assets/6895245/5e5eb3a8-ca13-4dda-92e2-43fe8e8dba94" width="500"/>


## パイプラインのコンパイルと実行

まず、パイプライン コンパイラに渡す環境変数を定義します。

In [ ]:
ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATA_ROOT = f"{ARTIFACT_STORE}/data"

TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"

TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
BASE_OUTPUT_DIR = f"{ARTIFACT_STORE}/models/{TIMESTAMP}"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT_ID={PROJECT_ID}
%env REGION={REGION}
%env SERVING_CONTAINER_IMAGE_URI={SERVING_CONTAINER_IMAGE_URI}
%env TRAINING_CONTAINER_IMAGE_URI={TRAINING_CONTAINER_IMAGE_URI}
%env TRAINING_FILE_PATH={TRAINING_FILE_PATH}
%env VALIDATION_FILE_PATH={VALIDATION_FILE_PATH}
%env BASE_OUTPUT_DIR={BASE_OUTPUT_DIR}

`ARTIFACT_STORE` が作成されていることを確認し、作成されていない場合は作成します。

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

**注:** アーティファクト ストアが事前かつ適切に作成・設定されていなかった場合は、Agent Platform からのアクセスを許可するために、**Cloud Shell** で次のコマンドを実行する必要がある場合があります。

```
PROJECT_ID=$(gcloud config get-value project)
PROJECT_NUMBER=$(gcloud projects list --filter="name=$PROJECT_ID" --format="value(PROJECT_NUMBER)")
gcloud projects add-iam-policy-binding $PROJECT_ID \
    --member="serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com" \
    --role="roles/storage.objectAdmin"
```

#### CLI コンパイラを使用したパイプラインのコンパイル

次のコマンドを使用して、生成した Python ファイルからパイプラインを JSON 記述にコンパイルします。

In [ ]:
PIPELINE_YAML = "covertype_kfp_pipeline.yaml"

In [ ]:
!kfp dsl compile --py pipeline/pipeline.py --output $PIPELINE_YAML

**注:** Python SDK を使用してパイプラインをコンパイルすることもできます。

```python
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=create_pipeline, 
    package_path=PIPELINE_YAML,
)

```

結果としてパイプライン ファイルが生成されます。

In [ ]:
!head {PIPELINE_YAML}

### パイプライン パッケージのデプロイと実行

In [ ]:
EXPERIMENT_NAME = "kfp-covertype-experiment"

In [ ]:
aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    experiment=EXPERIMENT_NAME,
    experiment_tensorboard=False,
)

pipeline = aiplatform.PipelineJob(
    display_name="covertype_kfp_pipeline_challenge_lab",
    template_path=PIPELINE_YAML,
    enable_caching=True,
)

pipeline.submit(experiment=EXPERIMENT_NAME)

### バッチ予測結果の確認
結果を確認してみましょう。予測結果として `prediction` カラムが設定されているはずです。

In [ ]:
%%bigquery

select * from covertype_dataset.batch_prediction_result ORDER BY rand() LIMIT 10

Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.